# Prepare the Data

In [ ]:
import pickle
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from sklearn import set_config
set_config(transform_output="pandas")

In [ ]:
train_set = pd.read_parquet("train_set.parquet")
X = train_set.drop(columns="early_spring")
y = train_set["early_spring"].astype(int)
X.shape, y.shape

In [ ]:
# Must be defined before loading preprocessor.pkl in another notebook
class EarlySpringFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, add_deltas=True, combine_oni=True, drop_redundant=True, use_year=False,
                 delta_features=("soil_temp_mean", "farm_avg_temp_mean", "om_temp_mean"),
                 redundant_features=("soil_thawed_days_jan", "soil_thawed_days_feb", "farm_freezing_days_feb")):
        self.add_deltas = add_deltas
        self.combine_oni = combine_oni
        self.drop_redundant = drop_redundant
        self.use_year = use_year
        self.delta_features = delta_features
        self.redundant_features = redundant_features

    def fit(self, X, y=None):
        self.feature_names_out_ = self._engineer(X).columns.to_numpy()
        return self

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_

    def transform(self, X):
        return self._engineer(X)

    def _engineer(self, X):
        data = X.drop(columns="early_spring", errors="ignore")
        data["phil_pred"] = data["phil_pred"].astype(int)

        if self.add_deltas:
            for name in self.delta_features:
                data[f"{name}_delta"] = data[f"{name}_feb"] - data[f"{name}_jan"]

        if self.combine_oni:
            data["oni_winter_avg"] = data[["oni_prev_nov", "oni_prev_dec"]].mean(axis=1)
            data = data.drop(columns=["oni_prev_nov", "oni_prev_dec"])

        if self.drop_redundant:
            data = data.drop(columns=list(self.redundant_features), errors="ignore")

        if self.use_year:
            data["year"] = data.index.to_numpy()

        return data

In [ ]:
SKEWED = r"(precip_sum|snowfall_sum|snowdepth_mean)"
skewed_columns = make_column_selector(pattern=SKEWED)
other_numeric_columns = make_column_selector(pattern=rf"^(?!phil_pred$)(?!.*{SKEWED}).*$")

log_transformer = FunctionTransformer(np.log1p, np.expm1, feature_names_out="one-to-one")

skewed_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("log", log_transformer),
    ("scale", StandardScaler()),
])

numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])

preprocessor = Pipeline(steps=[
    ("features", EarlySpringFeatures()),
    ("columns", ColumnTransformer(
        transformers=[
            ("phil", "passthrough", ["phil_pred"]),
            ("skewed", skewed_pipeline, skewed_columns),
            ("numeric", numeric_pipeline, other_numeric_columns),
        ],
        verbose_feature_names_out=False,
    )),
]).set_output(transform="pandas")
preprocessor

In [ ]:
X_prepared = preprocessor.fit_transform(X, y)
X_prepared.head()

In [ ]:
with open("preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)